# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hapepaAhmed/my-capstone-project/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# **LOAD THE Data**

In [1]:
!pip install -q huggingface_hub

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import userdata
from huggingface_hub import hf_hub_download

In [3]:
HF_TOKEN = userdata.get("HF_TOKEN")

In [4]:
parquet_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

In [5]:
df = pd.read_parquet(parquet_path)

print(df.shape)

(9841378, 30)


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector combines search visibility, traffic, and engagement metrics. Additional features are engineered from existing observations to capture user behavior while avoiding future information. All engineered features are computed using values available within the same analysis period.

In [6]:
import pandas as pd
import numpy as np

# Copy the original data
feature_df = df.copy()

# -----------------------------
# Engineered Features
# -----------------------------

# Click-Through Rate (CTR)
feature_df["ctr"] = np.where(
    feature_df["gsc_impressions"] > 0,
    feature_df["gsc_clicks"] / feature_df["gsc_impressions"],
    0
)

# Engagement Rate
feature_df["engagement_rate"] = np.where(
    feature_df["ga4_sessions"] > 0,
    feature_df["ga4_engaged_sessions"] / feature_df["ga4_sessions"],
    0
)

# Average engagement time per session
feature_df["avg_engagement_sec"] = np.where(
    feature_df["ga4_sessions"] > 0,
    feature_df["ga4_total_engagement_sec"] / feature_df["ga4_sessions"],
    0
)

#--------------------------
#Clicks per Session
#---------------------------
feature_df["clicks_per_session"] = np.where(
    feature_df["ga4_sessions"] > 0,
    feature_df["gsc_clicks"] /
    feature_df["ga4_sessions"],
    0
)

#--------------------------
#Scrolls per Session
#--------------------------
feature_df["scrolls_per_session"] = np.where(
    feature_df["ga4_sessions"] > 0,
    feature_df["scroll_events"] /
    feature_df["ga4_sessions"],
    0
)

# -----------------------------
# Categorical Handling
# -----------------------------

feature_df["position_bucket"] = pd.cut(
    feature_df["gsc_avg_position"],
    bins=[0, 10, 20, 50, float("inf")],
    labels=["Top10", "11-20", "21-50", "50+"]
)

# Encode the categorical feature
feature_df["position_bucket"] = (
    feature_df["position_bucket"]
    .cat.codes
)

# -----------------------------
# Final Feature Vector
# -----------------------------

feature_vector = feature_df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ctr",
        "engagement_rate",
        "avg_engagement_sec",
        "clicks_per_session",
        "scrolls_per_session",
        "position_bucket"
    ]
]

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [7]:
import pandas as pd

summary = pd.DataFrame({
    "Feature": feature_vector.columns,
    "Data Type": feature_vector.dtypes.astype(str).values,
    "Missing Count": feature_vector.isnull().sum().values,
})

summary["Missing %"] = (
    summary["Missing Count"] / len(feature_vector) * 100
).round(2)

summary["Unique Values"] = [
    feature_vector[col].nunique()
    for col in feature_vector.columns
]

summary["Mean"] = [
    feature_vector[col].mean()
    if pd.api.types.is_numeric_dtype(feature_vector[col])
    else None
    for col in feature_vector.columns
]

summary["Category"] = [
    "Categorical"
    if pd.api.types.is_object_dtype(feature_vector[col])
    or pd.api.types.is_categorical_dtype(feature_vector[col])
    else "Numerical"
    for col in feature_vector.columns
]

summary


/tmp/ipykernel_1093/598505892.py:28: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  or pd.api.types.is_categorical_dtype(feature_vector[col])


,Feature,Data Type,Missing Count,Missing %,Unique Values,Mean,Category
0,gsc_impressions,int64,0,0.00,4854,28.518119,Numerical
1,gsc_clicks,int64,0,0.00,140,0.083508,Numerical
2,gsc_avg_position,float64,6230317,63.31,457684,15.826651,Numerical
3,ctr,float64,0,0.00,15529,0.001130,Numerical
4,engagement_rate,float64,0,0.00,275,0.001464,Numerical
5,avg_engagement_sec,float64,0,0.00,5824,0.229394,Numerical
6,clicks_per_session,float64,0,0.00,956,0.017578,Numerical
7,scrolls_per_session,float64,0,0.00,1059,0.007741,Numerical
8,position_bucket,int8,0,0.00,5,-0.384165,Numerical


In [8]:
# Handle missing values

numeric_cols = feature_vector.select_dtypes(
    include="number"
).columns

feature_vector[numeric_cols] = (
    feature_vector[numeric_cols]
    .fillna(
        feature_vector[numeric_cols]
        .median()
    )
)

feature_vector["position_bucket"] = (
    feature_vector["position_bucket"]
    .fillna("Unknown")
)

/tmp/ipykernel_1093/3437679973.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  feature_vector[numeric_cols] = (
/tmp/ipykernel_1093/3437679973.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  feature_vector["position_bucket"] = (


# Feature Documentation

In [9]:
feature_notes = pd.DataFrame({
    "Feature":[
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ctr",
        "engagement_rate",
        "avg_engagement_sec",
        "clicks_per_session",
        "scrolls_per_session",
        "position_bucket"
    ],

    "Meaning":[
        "Organic search impressions",
        "Organic search clicks",
        "Average search position",
        "Clicks divided by impressions",
        "Engaged sessions divided by sessions",
        "Average engagement seconds",
        "Clicks per session",
        "Scroll events per session",
        "Grouped search position"
    ],

    "Missing Handling":[
        "Median",
        "Median",
        "Median",
        "Filled with 0",
        "Filled with 0",
        "Filled with 0",
        "Filled with 0",
        "Filled with 0",
        "Mode"
    ],

    "Available Before Prediction":[
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]
})

feature_notes

,Feature,Meaning,Missing Handling,Available Before Prediction
0,gsc_impressions,Organic search impressions,Median,Yes
1,gsc_clicks,Organic search clicks,Median,Yes
2,gsc_avg_position,Average search position,Median,Yes
3,ctr,Clicks divided by impressions,Filled with 0,Yes
4,engagement_rate,Engaged sessions divided by sessions,Filled with 0,Yes
5,avg_engagement_sec,Average engagement seconds,Filled with 0,Yes
6,clicks_per_session,Clicks per session,Filled with 0,Yes
7,scrolls_per_session,Scroll events per session,Filled with 0,Yes
8,position_bucket,Grouped search position,Mode,Yes


In [10]:
summary_after = pd.DataFrame({
    "Feature": feature_vector.columns,
    "Missing After Fill": feature_vector.isnull().sum().values
})

summary_after

,Feature,Missing After Fill
0,gsc_impressions,0
1,gsc_clicks,0
2,gsc_avg_position,0
3,ctr,0
4,engagement_rate,0
5,avg_engagement_sec,0
6,clicks_per_session,0
7,scrolls_per_session,0
8,position_bucket,0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [11]:
identifier_columns = [
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

identifier_leakage = [
    col
    for col in feature_vector.columns
    if col in identifier_columns
]

print(identifier_leakage)

[]


# **Leakage Assessment**

All engineered features are computed using information available within the same observation window. No future performance measurements, manually created labels, optimization outcomes, or identifier fields are included in the feature vector. Therefore, no evidence of data leakage was identified.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [12]:
excluded_features = pd.DataFrame({

    "Field":[
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "sessions_ai",
        "ai_chatgpt",
        "ai_perplexity",
        "ai_gemini",
        "ai_copilot",
        "ai_claude",
        "ai_meta",
        "ai_other"
    ],

    "Reason":[
        "Identifier",
        "Identifier",
        "Filtering only",
        "Outside project scope",
        "Outside project scope",
        "Outside project scope",
        "Outside project scope",
        "Outside project scope",
        "Outside project scope",
        "Outside project scope",
        "Outside project scope"
    ]

})

excluded_features


,Field,Reason
0,client_hash_id,Identifier
1,content_hash_id,Identifier
2,report_date,Filtering only
3,sessions_ai,Outside project scope
4,ai_chatgpt,Outside project scope
5,ai_perplexity,Outside project scope
6,ai_gemini,Outside project scope
7,ai_copilot,Outside project scope
8,ai_claude,Outside project scope
9,ai_meta,Outside project scope


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.